# Model Golf · ECG to ESP32
MassAI · November 18, 2026

Run the same small neural network in Python and on a simulated chip. You’ll compare the raw model score and predicted class for two identical ECG windows.

**In Colab:** Open this `.ipynb` with **File → Upload notebook**, leave the runtime on CPU, and run the cells in order. When prompted, upload `model-golf-data.zip` from the extracted student kit. In local Jupyter, keep that data ZIP beside the notebook. NumPy 2.2.6 was used for the reference build.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import sys

archive_name = 'model-golf-data.zip'
base = Path('.')
if not (base / 'classroom.npz').exists():
    if not (base / archive_name).exists():
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError(f'Put {archive_name} beside this notebook.') from error
        print(f'Choose {archive_name} in the upload dialog.')
        uploaded = files.upload()
        if archive_name not in uploaded:
            raise ValueError(f'Please upload the supplied {archive_name}.')
        (base / archive_name).write_bytes(uploaded[archive_name])
    base = Path('model-golf-data')
    members = ('classroom.npz', 'demo_cases.npz', 'baseline.mgf', 'model_golf.py',
               'rules.json', 'wokwi/diagram.json', 'wokwi/src/main.cpp', 'wokwi/src/inference.h')
    with ZipFile(archive_name) as archive:
        for name in members:
            target = base / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))

import numpy as np
sys.path.insert(0, str(base.resolve()))
from model_golf import (LABELS, WIDTHS, browser_sketch, decode, encode, firmware_headers,
                        format_score, logits, score, train)

with np.load(base / 'classroom.npz', allow_pickle=False) as data:
    x_train, y_train = data['train_x'], data['train_y']
    x_val, y_val = data['validation_x'], data['validation_y']
with np.load(base / 'demo_cases.npz', allow_pickle=False) as data:
    cases = {name: data[name] for name in ('id', 'x', 'y')}
baseline = (base / 'baseline.mgf').read_bytes()
print(f'{len(x_train):,} training windows · {len(x_val):,} practice windows · {x_train.shape[1]} values per window')

## The two labels

Each example is an MLII ECG window around an expert beat annotation in the MIT-BIH Arrhythmia Database. `N` is the database’s normal beat symbol. `V` is a premature ventricular contraction (PVC). The dataset omits other annotation symbols and uses different recordings for training, practice and final scoring. This classroom model classifies pre-cut windows for those two labels. Clinical use would require a separate validation process.

Start with the included working weights. Accuracy counts every correct prediction. PVC sensitivity counts the share of annotated PVC beats that the model catches.

[MIT-BIH data and license](https://physionet.org/content/mitdb/1.0.0/) · [Beat annotation codes](https://physionet.org/physiotools/wpg/wpg_36.htm)

In [ ]:
baseline_score = score(baseline, x_val, y_val)
print('Pretrained 24-neuron FP32 model')
print(format_score(baseline_score))
print(f'Always-N practice accuracy: {np.mean(y_val == 0):.1%} | PVC sensitivity: 0.0%')

In [ ]:
import matplotlib.pyplot as plt
import hashlib

reference = logits(decode(baseline), cases['x'])
time_ms = (3 * np.arange(64) - 62) * 1000 / 360
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True, layout='constrained')
for ax, beat_id, x, label, z, color in zip(axes, cases['id'], cases['x'], cases['y'],
                                            reference, ('#137e70', '#c4453a')):
    ax.plot(time_ms, x, color=color, linewidth=2)
    ax.axvline(0, color='0.5', linestyle=':')
    ax.set_title(f'{beat_id} · annotated {LABELS[label]}')
    ax.set_xlabel('Milliseconds from annotation')
    print(f'{beat_id} | logit {z:+.8f} | predicted {LABELS[int(z >= 0)]}')
axes[0].set_ylabel('Normalized amplitude')
plt.show()
print('Model SHA-256:', hashlib.sha256(baseline).hexdigest())

## Export the demo

The firmware uses the 64 processed values verbatim, runs both MLP layers, and switches the LEDs from its own logit. A nonnegative logit predicts PVC. The Python reference is embedded for the serial comparison.

Open a [new Wokwi ESP32 project](https://wokwi.com/projects/new/esp32). From the ZIP below, paste `sketch.ino` and `diagram.json` over the corresponding tabs. Start the simulation and look for `parity=PASS` on each beat. The project changes can run without saving to an account.

In [ ]:
import shutil

out = Path('baseline_export')
src = out / 'src'
firmware_headers(baseline, cases, src)
for name in ('main.cpp', 'inference.h'):
    shutil.copyfile(base / 'wokwi' / 'src' / name, src / name)
(out / 'sketch.ino').write_text(browser_sketch(src))
shutil.copyfile(base / 'wokwi' / 'diagram.json', out / 'diagram.json')
(out / 'baseline.mgf').write_bytes(baseline)
with ZipFile('baseline-wokwi.zip', 'w') as export:
    for name in ('baseline.mgf', 'sketch.ino', 'diagram.json'):
        export.write(out / name, name)
print('Saved baseline-wokwi.zip')

In [ ]:
# Run this cell in Colab to download the demo. In Jupyter use the Files panel.
try:
    from google.colab import files
except ImportError:
    print(Path('baseline-wokwi.zip').resolve())
else:
    files.download('baseline-wokwi.zip')

## Optional · Reproduce training

This cell trains from scratch with seed 18. Compare the practice metrics and demo logits. Floating-point libraries on different platforms can produce slightly different trained weights. Use the shipped payload for the reference firmware comparison above.

In [ ]:
retrained = encode(train(x_train, y_train, width=24), bits=32)
print(format_score(score(retrained, x_val, y_val)))
print('Retrained demo logits:', logits(decode(retrained), cases['x']))
Path('retrained-baseline.mgf').write_bytes(retrained)